### Reading CSV Files

This notebook covers every way to read CSV files in Python — from the built-in `csv`
module to the full power of `pandas.read_csv()`. Starting with the standard library
shows you what happens under the hood before library abstractions hide the details.

**Sections**
1. Standard Library: `csv` module
2. Pandas `pd.read_csv()` - organized by parameter groups
3. Writing CSV files back
4. Summary quick-reference table

## Setup

Run this cell first. It imports libraries and confirms the sample data files are present.

In [15]:
import os, csv, io
import pandas as pd

In [3]:
files = ['data/employees.csv', 'data/employees_pipe.csv', 'data/employees_messy.csv']
for f in files:
    print("OK " if os.path.exists(f) else "MISSING ", f)

OK  data/employees.csv
OK  data/employees_pipe.csv
OK  data/employees_messy.csv


### **Standard Library: The `csv` Module**

The built-in `csv` module handles reading and writing CSV without any dependencies.

### `csv.reader` - Basic Row Iteration

`csv.reader` returns each row as a plain list of strings. Key parameters:
- `delimiter` - field separator character (default `,`)
- `quotechar` - character used to wrap fields containing the delimiter (default `"`)
- `skipinitialspace` - strip leading whitespace after delimiter (default `False`)
- `escapechar` - character used to escape the quotechar when not using quoting

In [9]:
# Default usage: delimiter=',', quotechar='"'
print("Default csv.reader in Python: ")

with open('data/employees.csv', newline='') as f:
    reader = csv.reader(f)
    for i, row in enumerate(reader):
        print(row)
        if i == 3:
            break

Default csv.reader in Python: 
['name', 'age', 'city', 'salary', 'department', 'start_date']
['Alice', '30', 'New York', '85000', 'Engineering', '2020-01-15']
['Bob', '25', 'San Francisco', '72000', 'Marketing', '2021-06-01']
['Charlie', '35', 'Chicago', '95000', 'Engineering', '2019-03-20']


In [11]:
# Custom delimiter
print("Reading Pipe-delimited file: ")

with open('data/employees_pipe.csv', newline='') as f:
    reader = csv.reader(f, delimiter='|')
    for row in reader:
        print(row)

Reading Pipe-delimited file: 
['name', 'age', 'city', 'salary']
['Alice', '30', 'New York', '85000']
['Bob', '25', 'San Francisco', '72000']
['Charlie', '35', 'Chicago', '95000']


In [19]:
# skipinitialspace demo for data with spaces after commas
sample = "name, age, city\nAlice, 30, New York\nBob, 25, Boston"

print("skipinitialspace=False (default): ")
reader = csv.reader(io.StringIO(sample), skipinitialspace=False)
for row in reader:
    print(row)

skipinitialspace=False (default): 
['name', ' age', ' city']
['Alice', ' 30', ' New York']
['Bob', ' 25', ' Boston']


Look at the words like age, city, New York etc. There is a space before these words, because skipintialspace = False. This issue is being fixed below.

In [28]:
print("skipinitialspace = True: ")
reader = csv.reader(io.StringIO(sample), skipinitialspace=True)
for row in reader:
    print(row)

skipinitialspace = True: 
['name', 'age', 'city']
['Alice', '30', 'New York']
['Bob', '25', 'Boston']


### `csv.DictReader` - Rows as Dictionaries

`csv.DictReader` maps each row to a `dict` using the header row as keys.
Pass `fieldnames` to override or supply headers when the file has none.

In [31]:
print("csv.DictReader: ")

with open('data/employees.csv', newline='') as f:
    reader = csv.DictReader(f)
    print("Fields:", reader.fieldnames)
    for i, row in enumerate(reader):
        print(dict(row))
        if i == 2:
            break

csv.DictReader: 
Fields: ['name', 'age', 'city', 'salary', 'department', 'start_date']
{'name': 'Alice', 'age': '30', 'city': 'New York', 'salary': '85000', 'department': 'Engineering', 'start_date': '2020-01-15'}
{'name': 'Bob', 'age': '25', 'city': 'San Francisco', 'salary': '72000', 'department': 'Marketing', 'start_date': '2021-06-01'}
{'name': 'Charlie', 'age': '35', 'city': 'Chicago', 'salary': '95000', 'department': 'Engineering', 'start_date': '2019-03-20'}


In [33]:
# Supplying custom fieldnames (file with no header row)
headerless = "Alice,30,New York\nBob,25,Boston"
reader = csv.DictReader(
    io.StringIO(headerless),
    fieldnames=['name', 'age', 'city'])

for row in reader:
    print(dict(row))

{'name': 'Alice', 'age': '30', 'city': 'New York'}
{'name': 'Bob', 'age': '25', 'city': 'Boston'}


### **Pandas `pd.read_csv()`**

`pd.read_csv()` is the most common way to load CSV data. Its parameters are organized
below by what they control, understanding the groups of parameters makes the docs much easier to navigate.

### Group 1 - File & Path Parameters

| Parameter | Type | Default | What it does |
|-----------|------|---------|--------------|
| `filepath_or_buffer` | str / Path / URL | required | File path, URL, or file-like object |
| `sep` / `delimiter` | str | `','` | Field separator (use `sep=r'\s+'` for whitespace) |
| `encoding` | str | `'utf-8'` | File character encoding |

In [38]:
# Basic read — default separator, utf-8 encoding
df = pd.read_csv('data/employees.csv')
print("Shape:", df.shape)
df.head(3)

Shape: (7, 6)


,name,age,city,salary,department,start_date
0,Alice,30.0,New York,85000,Engineering,2020-01-15
1,Bob,25.0,San Francisco,72000,Marketing,2021-06-01
2,Charlie,35.0,Chicago,95000,Engineering,2019-03-20


In [40]:
# Custom delimiter
df_pipe = pd.read_csv('data/employees_pipe.csv', sep='|')

print("Pipe-delimited:")
df_pipe

Pipe-delimited:


,name,age,city,salary
0,Alice,30,New York,85000
1,Bob,25,San Francisco,72000
2,Charlie,35,Chicago,95000


### Group 2 - Row & Column Selection

| Parameter | Type | Default | What it does |
|-----------|------|---------|--------------|
| `header` | int / list / None | `0` | Row number(s) to use as column names; `None` = no header |
| `index_col` | int / str / list / False | `None` | Column(s) to use as row index |
| `usecols` | list / callable | `None` | Load only these columns |
| `skiprows` | int / list / callable | `None` | Rows to skip at the start |
| `nrows` | int | `None` | Maximum rows to read |
| `skipfooter` | int | `0` | Rows to skip at the end (requires `engine='python'`) |

In [43]:
# usecols to load only name and salary
df_subset = pd.read_csv('data/employees.csv', usecols=['name', 'salary'])
print(df_subset.head(3)) #dataframe with only name and salary column

# usecols with callable for columns whose name starts with a vowel
df_vowel = pd.read_csv('data/employees.csv', usecols=lambda c: c[0].lower() in 'aeiou')

print("\nColumns starting with a vowel:", df_vowel.columns.tolist())

      name  salary
0    Alice   85000
1      Bob   72000
2  Charlie   95000

Columns starting with a vowel: ['age']


In [45]:
# skiprows + nrows
# skiprows=2 skips the first 2 physical rows (the header + 1 data row),
# nrows=3 reads the next 3 rows, header=None means no auto-detected header

df_partial = pd.read_csv('data/employees.csv', skiprows=2, nrows=3, header=None)
print("Skipped header along with the first data row (Alice) and loaded next 3 rows: ")
print(df_partial)

Skipped header along with the first data row (Alice) and loaded next 3 rows: 
         0   1              2      3            4           5
0      Bob  25  San Francisco  72000    Marketing  2021-06-01
1  Charlie  35        Chicago  95000  Engineering  2019-03-20
2    Diana  28       New York  68000           HR  2022-02-10


In [47]:
# header=None when file has no header row
headerless = "Alice,30,New York\nBob,25,Boston"

df_nh = pd.read_csv(io.StringIO(headerless), header=None, names=['name', 'age', 'city'])
print(df_nh)

    name  age      city
0  Alice   30  New York
1    Bob   25    Boston


### Group 3 - Data Type Control

| Parameter | Type | Default | What it does |
|-----------|------|---------|--------------|
| `dtype` | dict / str | `None` | Force specific dtype per column or for all columns |
| `parse_dates` | bool / list / dict | `False` | Parse columns as datetime |
| `na_values` | str / list / dict | `None` | Additional values to treat as NaN |
| `keep_default_na` | bool | `True` | Whether pandas' built-in NA list applies |
| `true_values` / `false_values` | list | `None` | Values to interpret as True/False |

In [50]:
# Pandas infers dtypes of each column by default
df = pd.read_csv('data/employees.csv')
print("Default dtypes:")
df.dtypes

Default dtypes:


name           object
age           float64
city           object
salary          int64
department     object
start_date     object
dtype: object

In [52]:
df

,name,age,city,salary,department,start_date
0,Alice,30.0,New York,85000,Engineering,2020-01-15
1,Bob,25.0,San Francisco,72000,Marketing,2021-06-01
2,Charlie,35.0,Chicago,95000,Engineering,2019-03-20
3,Diana,28.0,New York,68000,HR,2022-02-10
4,Eve,32.0,Boston,88000,Engineering,2020-11-05
5,Frank,NaN,Austin,76000,Marketing,2021-09-15
6,Grace,29.0,New York,91000,Engineering,2020-07-01


In [54]:
# Force salary to float, age to Int64 (nullable integer)
df_typed = pd.read_csv('data/employees.csv', dtype={'salary': float, 'age': 'Int64'})
print("With explicit dtype:")
print(df_typed.dtypes)
print()

With explicit dtype:
name           object
age             Int64
city           object
salary        float64
department     object
start_date     object
dtype: object



In [56]:
df_typed

,name,age,city,salary,department,start_date
0,Alice,30,New York,85000.0,Engineering,2020-01-15
1,Bob,25,San Francisco,72000.0,Marketing,2021-06-01
2,Charlie,35,Chicago,95000.0,Engineering,2019-03-20
3,Diana,28,New York,68000.0,HR,2022-02-10
4,Eve,32,Boston,88000.0,Engineering,2020-11-05
5,Frank,<NA>,Austin,76000.0,Marketing,2021-09-15
6,Grace,29,New York,91000.0,Engineering,2020-07-01


In [58]:
# parse_dates
df_dates = pd.read_csv('data/employees.csv', parse_dates=['start_date'])

print("start_date dtype:", df_dates['start_date'].dtype)
print(df_dates[['name', 'start_date']].head(3))

start_date dtype: datetime64[ns]
      name start_date
0    Alice 2020-01-15
1      Bob 2021-06-01
2  Charlie 2019-03-20


In [60]:
# "na_values" to treat empty string and "N/A" as NaN
df_na = pd.read_csv('data/employees.csv', na_values=['', 'N/A', 'unknown'])

print("Missing values per column:")
df_na.isna().sum()

Missing values per column:


name          0
age           1
city          0
salary        0
department    0
start_date    0
dtype: int64

### Group 4 - Performance Parameters

| Parameter | Type | Default | What it does |
|-----------|------|---------|--------------|
| `chunksize` | int | `None` | Return an iterator of DataFrames with this many rows each |
| `iterator` | bool | `False` | Return a `TextFileReader` object for manual chunking |
| `engine` | str | `'c'` | Parser engine: `'c'` (fast), `'python'` (feature-rich), `'pyarrow'` (fastest for large files) |
| `memory_map` | bool | `False` | Memory-map the file for faster access on disk-backed files |
| `low_memory` | bool | `True` | Process file in chunks internally to reduce memory (may cause mixed-type columns) |

In [63]:
# "chunksize" to process large files in pieces without loading all into RAM
total_salary = 0
row_count = 0

for chunk in pd.read_csv('data/employees.csv', chunksize=3):
    total_salary += chunk['salary'].sum()
    row_count += len(chunk)

print(f"Processed {row_count} rows in chunks")
print(f"Total salary: ${total_salary:,.0f}")
print(f"Average salary: ${total_salary/row_count:,.0f}")

Processed 7 rows in chunks
Total salary: $575,000
Average salary: $82,143


In [71]:
# engine comparison
import time

# C engine (default, fastest)
start = time.perf_counter()
df_c = pd.read_csv('data/employees.csv', engine='c')
c_time = time.perf_counter() - start

In [73]:
# Python engine (slower, but supports more features like skipfooter)
start = time.perf_counter()
df_py = pd.read_csv('data/employees.csv', engine='python')
py_time = time.perf_counter() - start

In [75]:
print(f"C engine: {c_time*1000:.3f} ms")
print(f"Python engine: {py_time*1000:.3f} ms")
print("(difference is negligible on small files - matters at millions of rows)")

C engine: 4.154 ms
Python engine: 2.326 ms
(difference is negligible on small files - matters at millions of rows)


### Group 5 - Edge Case Parameters

| Parameter | Type | Default | What it does |
|-----------|------|---------|--------------|
| `on_bad_lines` | str | `'error'` | What to do with lines that have too many fields: `'error'`, `'warn'`, `'skip'` |
| `comment` | str | `None` | Character marking comment lines to ignore |
| `thousands` | str | `None` | Thousands separator (e.g. `','` to parse `"85,000"` as `85000`) |
| `decimal` | str | `'.'` | Decimal separator (e.g. `','` for European format) |
| `lineterminator` | str | `None` | Custom line terminator |
| `quotechar` | str | `'"'` | Character used to denote start/end of quoted field |
| `escapechar` | str | `None` | Escape character for quotechar |

In [78]:
# comment='#' skips lines starting with # 
# thousands=',' parses "85,000" as 85000
# on_bad_lines='skip' ignores malformed rows
df_skip = pd.read_csv(
    'data/employees_messy.csv',
    comment='#',
    thousands=',',
    on_bad_lines='skip'
)

print("Loaded after skipping bad lines:")
print(df_skip)

Loaded after skipping bad lines:
                             name   age  salary_usd
0                           Alice  30.0     85000.0
1                             Bob  25.0     72000.0
2                           Diana  28.0     68000.0
3  BAD LINE WITHOUT ENOUGH FIELDS   NaN         NaN
4                             Eve  32.0     88000.0


In [80]:
# on_bad_lines='warn' emits a warning instead of silently skipping
import warnings
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    df_warn = pd.read_csv(
        'data/employees_messy.csv',
        comment='#',
        thousands=',',
        on_bad_lines='warn'
    )
    if w:
        print("Warning raised:", w[0].message)
    else:
        print("No warning raised (pandas version may handle this differently)")
print(df_warn)

No warning raised (pandas version may handle this differently)
                             name   age  salary_usd
0                           Alice  30.0     85000.0
1                             Bob  25.0     72000.0
2                           Diana  28.0     68000.0
3  BAD LINE WITHOUT ENOUGH FIELDS   NaN         NaN
4                             Eve  32.0     88000.0


### **Writing to csv files using `DataFrame.to_csv()`**

| Parameter | Default | What it does |
|-----------|---------|--------------|
| `path_or_buf` | `None` | File path; if `None` returns string |
| `sep` | `','` | Separator character |
| `index` | `True` | Write row index; usually pass `False` |
| `header` | `True` | Write column names |
| `columns` | `None` | Subset of columns to write |
| `encoding` | `'utf-8'` | Output encoding |
| `compression` | `'infer'` | Write compressed: `'gzip'`, `'zip'`, etc. |
| `na_rep` | `''` | String to use for missing values |
| `float_format` | `None` | Format string for floats, e.g. `'%.2f'` |
| `date_format` | `None` | Format string for datetime objects |

In [84]:
import pandas as pd

df = pd.read_csv('data/employees.csv', parse_dates=['start_date'])

# Basic write — without row index
df.to_csv('data/output_pandas.csv', index=False)

In [86]:
# Write specific columns, control float format and date format
df.to_csv(
    'data/output_formatted.csv',
    index=False,
    columns=['name', 'salary', 'start_date'],
    float_format='%.2f',
    date_format='%Y/%m/%d',
    na_rep='N/A',
)

In [88]:
# Write to gzip-compressed file
df.to_csv('data/employees.csv.gz', index=False, compression='gzip')
print("Files written:")

for f in ['output_pandas.csv', 'output_formatted.csv', 'employees.csv.gz']:
    size = os.path.getsize(f'data/{f}')
    print(f"  data/{f} - {size} bytes")

Files written:
  data/output_pandas.csv - 367 bytes
  data/output_formatted.csv - 182 bytes
  data/employees.csv.gz - 257 bytes


In [90]:
# to_csv with no path returns string (useful for previewing)
snippet = df.head(2).to_csv(index=False)
print(snippet)

name,age,city,salary,department,start_date
Alice,30.0,New York,85000,Engineering,2020-01-15
Bob,25.0,San Francisco,72000,Marketing,2021-06-01

